## Build Unique Concrete Plant List from EC3 EPD Data

Loads the cleaned EPD dataset and extracts a deduplicated list of concrete plants
with their coordinates and EPD counts.

In [22]:
import pandas as pd

df = pd.read_pickle('../02_processed_data/epd_data_cleaned_all.pkl')
print(f"Total EPDs loaded: {len(df)}")
print(f"\nAvailable plant_or_group columns:")
print([c for c in df.columns if 'plant_or_group' in c])

Total EPDs loaded: 69305

Available plant_or_group columns:
['plant_or_group.name', 'plant_or_group.longitude', 'plant_or_group.created_on', 'plant_or_group.updated_on', 'plant_or_group.id', 'plant_or_group.latitude', 'plant_or_group.type']


### Check for duplicate coordinates under different plant names

Before deduplicating on `plant_or_group.name`, verify whether any (lat, lon) pair
maps to more than one distinct plant name — which would indicate data quality issues.

In [23]:
# Work with the relevant columns only
plant_cols = ['plant_or_group.name', 'plant_or_group.latitude', 'plant_or_group.longitude']
df_plants = df[plant_cols].copy()

# Drop rows missing any of the three fields
before = len(df_plants)
df_plants.dropna(subset=plant_cols, inplace=True)
print(f"Rows with complete plant name + coordinates: {len(df_plants)} (dropped {before - len(df_plants)} missing)")

# Strip leading/trailing whitespace from plant names
df_plants['plant_or_group.name'] = df_plants['plant_or_group.name'].str.strip()

# Also apply to the source dataframe so EPD counts use the cleaned names
df['plant_or_group.name'] = df['plant_or_group.name'].str.strip()

Rows with complete plant name + coordinates: 69305 (dropped 0 missing)


In [24]:
# For each (lat, lon) pair, count distinct plant names
coord_to_names = (
    df_plants
    .groupby(['plant_or_group.latitude', 'plant_or_group.longitude'])['plant_or_group.name']
    .nunique()
    .reset_index()
    .rename(columns={'plant_or_group.name': 'distinct_names'})
)

conflicts = coord_to_names[coord_to_names['distinct_names'] > 1]

if conflicts.empty:
    print("No conflicts: every (lat, lon) pair maps to exactly one plant name.")
else:
    print(f"Found {len(conflicts)} (lat, lon) pairs with multiple plant names.")
    print(f"These affect {conflicts['distinct_names'].sum()} total name entries.\n")

    # Show the conflicting records
    conflict_coords = conflicts.merge(
        df_plants.drop_duplicates(subset=plant_cols),
        on=['plant_or_group.latitude', 'plant_or_group.longitude']
    )
    print(conflict_coords.sort_values(['plant_or_group.latitude', 'plant_or_group.longitude']).to_string(index=False))

Found 43 (lat, lon) pairs with multiple plant names.
These affect 92 total name entries.

 plant_or_group.latitude  plant_or_group.longitude  distinct_names                plant_or_group.name
               21.378280               -157.899095               2                    Halawa Plant #1
               21.378280               -157.899095               2                    Halawa Plant #3
               26.292849                -81.793055               2         Naples Wiggings Pass RM/BM
               26.292849                -81.793055               2       Wiggings Pass (North Naples)
               29.620418                -95.543590               3                           Downtown
               29.620418                -95.543590               3              Missouri City RM (JV)
               29.620418                -95.543590               3                           Rothwell
               30.428317                -89.079589               2                            

### Build unique plant list with EPD counts

Group by `plant_or_group.name` and take the first lat/lon for each name
(they should be consistent for a given plant). Add the EPD count per plant.

In [25]:
# Count EPDs per plant name using the full (non-dropped) dataframe
epd_counts = (
    df.dropna(subset=['plant_or_group.name'])
    .groupby('plant_or_group.name')
    .size()
    .reset_index(name='epd_count')
)

# Get unique name -> coords (first occurrence of lat/lon per name)
unique_plants = (
    df_plants
    .drop_duplicates(subset=['plant_or_group.name'])
    [['plant_or_group.name', 'plant_or_group.latitude', 'plant_or_group.longitude']]
)

# Merge in EPD counts
active_plants = unique_plants.merge(epd_counts, on='plant_or_group.name', how='left')

# Rename columns for the output
active_plants.rename(columns={
    'plant_or_group.name': 'plant_name',
    'plant_or_group.latitude': 'latitude',
    'plant_or_group.longitude': 'longitude'
}, inplace=True)

active_plants = active_plants.sort_values('epd_count', ascending=False).reset_index(drop=True)

print(f"Unique concrete plants: {len(active_plants)}")
active_plants.head(10)

Unique concrete plants: 1211


,plant_name,latitude,longitude,epd_count
0,Newark,40.749200,-74.167230,1983
1,Roseland,40.826000,-74.297400,1974
2,Bogota,40.879960,-74.037690,1964
3,Howell,40.214260,-74.191050,1908
4,North Bergen,40.820150,-74.015510,1893
5,Broadway,40.737890,-74.079320,1717
6,Port of Portland,45.559590,-122.714160,1469
7,West Nyack,41.110050,-73.955760,1466
8,Vernon,34.014213,-118.224394,1133
9,Queens Lane (wet),37.370440,-121.903374,1062


### Filter down to plants only within the continental US

Uses shapefile to find plants within the US boundary

In [ ]:
import geopandas as gpd

# Load Natural Earth 110m countries shapefile (stored in 01_raw_data/)
world = gpd.read_file("../01_raw_data/country_shapefiles")
us_boundary = world.loc[world['ISO_A3'] == 'USA', 'geometry'].iloc[0]

# Convert plant locations to a GeoDataFrame for point-in-polygon test
gdf = gpd.GeoDataFrame(
    active_plants,
    geometry=gpd.points_from_xy(active_plants['longitude'], active_plants['latitude']),
    crs='EPSG:4326'
)

# 1. Point-in-polygon: keep only plants inside the US boundary
#    This correctly handles the irregular US-Canada border (e.g. excludes Victoria, BC at 48.4°N)
# 2. Bounding box clip: drop Alaska and Hawaii, restrict to continental US
active_plants = (
    gdf[gdf.within(us_boundary)]
    .query("24.5 <= latitude <= 49.5 and -125 <= longitude <= -66.5")
    .drop(columns='geometry')
    .reset_index(drop=True)
)

print(f"Continental US concrete plants after filtering: {len(active_plants)}")

In [27]:
# Save to CSV
active_plants.to_csv('../02_processed_data/active_concrete_plants_ec3.csv', index=False)
print("Saved to ../02_processed_data/active_concrete_plants_ec3.csv")

Saved to ../02_processed_data/active_concrete_plants_ec3.csv
